# Sandbox — poke the model

A freeform surface for reverse-engineering a circuit by hand. The study template runs a fixed protocol; this is for the exploratory loop that comes *before* you know which protocol to run — look at a head, break it, watch what changes, form a guess, test it in the next cell.

Each section is a self-contained recipe. Copy a cell, change the head or the checkpoint, re-run. Nothing here is sacred; overwrite it. The only rule the tooling enforces for you is that every ablation restores itself, so you can never leave a head silently zeroed.

The lab API in one screen:
```python
m = lab.load_checkpoint('nano_uniform')      # any family stem or path
lab.list_checkpoints()                        # what's on disk
logits, cache = m.model.run_with_cache(tok)   # 56 cache keys: L{i}.attn.{scores,q,pattern,z}, resid_*
with lab.ablate_heads(m.model, [(1,3)]): ...  # zero-ablate, auto-restore
with lab.resample_heads(m.model, [(1,3)], src_tok): ...  # on-manifold baseline
```

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, torch
import matplotlib.pyplot as plt
import gfmcd.lab as lab
from gfmcd.lab import viz

print("checkpoints:", lab.list_checkpoints())

## Load a model
Swap the stem to study anything on disk. `.pool` tells you the data regime it was trained on.

In [ ]:
m = lab.load_checkpoint("nano_uniform")
m   # repr shows family (mlm/causal), head (aa/lm), layers x heads, device

## Recipe 1 — look at one head's attention
Pick a head, run a few sequences, and look at what it attends to. Here: does L1H3 attend the start codon? Change `(L, H)` and re-run to inspect any head.

In [ ]:
from gfmcd.signatures import start_index
tok, aa, phase = lab.load_uniform_windows(n=32, seed=1)
start = start_index(phase)

L, H = 1, 3
with torch.no_grad():
    _, cache = m.model.run_with_cache(torch.from_numpy(tok).long().to(m.device))
pattern = cache[f"L{L}.attn.pattern"][:, H].cpu().numpy()   # [batch, q, k]

row = 0
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(pattern[row], cmap="magma", aspect="auto")
ax.axvline(start[row], color="c", lw=0.8, ls="--", label="start codon")
ax.set_xlabel("key (attended-to)"); ax.set_ylabel("query"); ax.set_title(f"L{L}H{H} attention, seq {row}")
ax.legend(); plt.colorbar(im, ax=ax, fraction=0.046); plt.show()

# how much does this head attend to the start codon, from downstream positions?
print("attention to start (this head):", float(lab.attention_survey(m, tok, start)[L, H]))

## Recipe 2 — ablate and diff, live
The core move. Measure something, turn a head off, measure again. Δloss is the arbiter — a head can carry huge attention and move the loss by nothing (the L1H2 lesson). Try `(1,2)` vs `(1,3)`.

In [ ]:
base_loss, base_acc = lab.aa_loss_acc(m, tok, aa)
print(f"intact:  loss {base_loss:.4f}  acc {base_acc:.4f}")

for (L, H) in [(1, 3), (1, 2)]:
    with lab.ablate_heads(m.model, (L, H)):
        loss, acc = lab.aa_loss_acc(m, tok, aa)
    print(f"ablate L{L}H{H}:  Δloss {loss-base_loss:+.5f}   Δacc {acc-base_acc:+.5f}   "
          f"({'ESSENTIAL' if loss-base_loss > 0.1 else 'inert'})")

## Recipe 3 — resample ablation (on-manifold baseline)
Zero-ablation pushes the residual stream off-distribution. Resample ablation patches the head's output from a *shuffled* batch instead — the head still writes, just a plausible value. Compare the two: a large zero-vs-resample gap means part of the 'effect' was distribution shock, not the head's computation.

In [ ]:
L, H = 1, 3
idx = torch.randperm(tok.shape[0])
src = torch.from_numpy(tok[idx]).long()

with lab.ablate_heads(m.model, (L, H)):
    zero_loss, _ = lab.aa_loss_acc(m, tok, aa)
with lab.resample_heads(m.model, [(L, H)], src):
    resample_loss, _ = lab.aa_loss_acc(m, tok, aa)

print(f"intact       {base_loss:.4f}")
print(f"zero-ablate  {zero_loss:.4f}  (Δ {zero_loss-base_loss:+.4f})")
print(f"resample     {resample_loss:.4f}  (Δ {resample_loss-base_loss:+.4f})")
print("gap between the two baselines:", f"{abs(zero_loss-resample_loss):.4f}")

## Recipe 4 — one cue-conflict example, read the probabilities
Insert a single nucleotide after the ATG so the anchor and the local content disagree, then read the model's amino-acid distribution at a far-downstream position. An anchor-relative circuit still reads the frame the anchor implies; a bag has drifted to the content frame. This is the mechanism test on one sequence, by hand.

In [ ]:
from gfmcd.signatures import frameshift
tok1, aa1, phase1 = lab.load_uniform_windows(n=4, seed=7)
start1 = start_index(phase1)

row = 0
# frameshift inserts one nt just after the ATG, so anchor and content now disagree downstream.
shifted = frameshift(tok1[row:row+1], np.array([start1[row] + 2]))
pos = start1[row] + 90   # far downstream

with torch.no_grad():
    p_intact  = m.model(torch.from_numpy(tok1[row:row+1]).long().to(m.device)).softmax(-1)[0, pos]
    p_shifted = m.model(torch.from_numpy(shifted).long().to(m.device)).softmax(-1)[0, pos]

top_intact  = int(p_intact.argmax());  top_shifted = int(p_shifted.argmax())
print(f"at position {pos} (distance {pos-start1[row]} from ATG):")
print(f"  intact:  argmax aa {top_intact}  p={float(p_intact[top_intact]):.3f}")
print(f"  shifted: argmax aa {top_shifted}  p={float(p_shifted[top_shifted]):.3f}")
print("  an anchor-relative circuit re-reads the frame from the moved anchor; a bag would not.")

## Recipe 5 — compare two checkpoints (e.g. across grokking)
The uniform anchor-relative frame circuit groks abruptly (~step 3000). Load an early and a late checkpoint of the same family and watch a circuit metric jump. Pin a step with `step=` (or pass the full filename).

In [ ]:
early = lab.load_checkpoint("nano_uniform", step=1200)
late  = lab.load_checkpoint("nano_uniform", step=12000)
tokc, aac, phc = lab.load_uniform_windows(n=64, seed=3)

for tag, mm in [("early (1200)", early), ("late (12000)", late)]:
    _, acc = lab.aa_loss_acc(mm, tokc, aac)
    sig = lab.mechanism_signature(mm, tokc, phc)
    print(f"{tag:14s}  aa_acc {acc:.4f}   anchor-share y={sig.anchor_share:.3f}   -> {sig.verdict}")

## Recipe 6 — the copy circuit, same tools
The induction models are causal with a nucleotide head. Everything above works; the assay names swap to their copy variants. One cell to see the whole circuit.

In [ ]:
mc = lab.load_checkpoint("repeat_copy_s0")
batch = lab.load_repeat_windows(n=64)
print(mc)
print("copy metrics:", {k: round(v,4) for k,v in lab.copy_metrics(mc, batch).items()})

attn = lab.copy_source_attention(mc, batch)
order = np.argsort(-attn.ravel())[:5]
print("top source heads:", [(f"L{k//mc.cfg.n_heads}H{k%mc.cfg.n_heads}", round(float(attn.ravel()[k]),3)) for k in order])
viz.plot_attention_survey(attn, title="repeat_copy_s0: attention to source"); plt.show()

## Scratch
Yours. Add cells below. The lab API is stable; the model is not going to bite.

In [ ]:
# scratch